# CNN path following

This notebook implements the ResNet18 model to determine what direction should be inputted at that moment

Automated labeling to isolate the bottom third of every frame and apply a HSV threshold to isolate the yellow path and find the centroid and putting the coordinates in the name of the file

In [ ]:
import cv2
import numpy as np
import os
import uuid

# Create a directory to store our training data
DATASET_DIR = 'f320951_dataset_left_third'
if not os.path.exists(DATASET_DIR):
    os.makedirs(DATASET_DIR)

# Open the video file
cap = cv2.VideoCapture('f320951_train14.mp4')
frame_count = 0

print("Starting Auto-Labeling...")
while True:
    ret, frame = cap.read()
    if not ret:
        break 
    height, width, _ = frame.shape
    
    # Isolate the bottom third of the image as the region of interest 
    roi_top = int(height * 2 / 3)
    roi = frame[roi_top:height, 0:width]
    
    # Yellow mask
    hsv = cv2.cvtColor(roi, cv2.COLOR_BGR2HSV)
    mask = cv2.inRange(hsv, (10, 60, 60), (45, 255, 255))
    
    # Find the centroid (center of mass) of the yellow pixels
    M = cv2.moments(mask)
    if M["m00"] > 0:
        # Calculate pixel coordinates
        cx = int(M["m10"] / M["m00"])
        cy = int(M["m01"] / M["m00"]) + roi_top 
        
        # Normalize coordinates to a range of [-1.0, 1.0] for the neural network
        # 0.0, 0.0 is the exact center of the image
        x_norm = (cx / width) * 2.0 - 1.0
        y_norm = (cy / height) * 2.0 - 1.0
        
        # Save image with JetBot naming convention: xy_<x>_<y>_<uuid>.jpg
        img_name = f"xy_{x_norm:.3f}_{y_norm:.3f}_{uuid.uuid1()}.jpg"
        img_path = os.path.join(DATASET_DIR, img_name)
        cv2.imwrite(img_path, frame)
        frame_count += 1

cap.release()
print(f"Auto-Labeling complete. Saved {frame_count} labeled images to '{DATASET_DIR}'.")

Starting Auto-Labeling...
Auto-Labeling complete. Saved 863 labeled images to 'less_bright'.


Preprocessing and loading data into PyTorch data loader

In [ ]:
import torch
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader, random_split
import glob
import os
from PIL import Image

DATASET_DIR = 'f320951_dataset_left_third'
if not os.path.exists(DATASET_DIR):
    os.makedirs(DATASET_DIR)

class XYDataset(Dataset):
    def __init__(self, directory):
        self.image_paths = glob.glob(os.path.join(directory, '*.jpg'))
        # Define the transformations to apply to each image
        self.transform = transforms.Compose([
            transforms.Resize((224, 224), antialias=True),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert('RGB')
        image = self.transform(image)
        # Extract x and y from the filename (e.g., xy_0.123_-0.456_uuid.jpg)
        filename = os.path.basename(img_path)
        parts = filename.split('_')
        x = float(parts[1])
        y = float(parts[2])
        
        return image, torch.tensor([x, y], dtype=torch.float32)

# Load the full dataset
dataset = XYDataset(DATASET_DIR)

# Calculate the 80/20 split sizes
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
# Split the dataset randomly
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

# Create two separate DataLoaders
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False, pin_memory=True) # Shuffle=False for validation

print(f"Dataset split: {len(train_dataset)} training images, {len(val_dataset)} validation images.")

Dataset split: 5353 training images, 1339 validation images.


ResNet18 architecture initialisation 

In [ ]:
import torchvision
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on device: {device}")

# Load the pre-trained model
model = torchvision.models.resnet18(weights='IMAGENET1K_V1')

# Replace the final classification layer with a regression layer (2 outputs: x, y)
model.fc = torch.nn.Linear(512, 2) 
model = model.to(device)

# SmoothL1Loss is much more robust to outliers than MSE
criterion = torch.nn.SmoothL1Loss()

# AdamW converges faster and handles weight decay better than standard SGD
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)
print("Model architecture modified for regression and sent to GPU.")

Training on device: cuda
Model architecture modified for regression and sent to GPU.


Model training and validation

In [ ]:
import torch

# Define the device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on device: {device}")

# Move the model to the chosen device
model = model.to(device)
epochs = 10 
print("Starting training...")

# Training loop
for epoch in range(epochs):
    model.train() 
    running_train_loss = 0.0
    
    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_train_loss += loss.item()

    avg_train_loss = running_train_loss / len(train_loader)
    
    # Validation phase
    model.eval() 
    running_val_loss = 0.0
    
    with torch.no_grad():
        for val_images, val_labels in val_loader:
            val_images = val_images.to(device)
            val_labels = val_labels.to(device)
            
            val_outputs = model(val_images)
            val_loss = criterion(val_outputs, val_labels)
            running_val_loss += val_loss.item()
    avg_val_loss = running_val_loss / len(val_loader)
    
    # Print both losses 
    print(f"Epoch [{epoch+1}/{epochs}] - Train Loss: {avg_train_loss:.4f} | Validation Loss: {avg_val_loss:.4f}")

# Save the trained weights 
torch.save(model.state_dict(), '30.4.26_third.pth')
print("Training complete! Model saved as '30.4.26_third.pth'")

Training on device: cuda
Starting training...
Epoch [1/10] - Train Loss: 0.0257 | Validation Loss: 0.0016
Epoch [2/10] - Train Loss: 0.0019 | Validation Loss: 0.0028
Epoch [3/10] - Train Loss: 0.0024 | Validation Loss: 0.0010
Epoch [4/10] - Train Loss: 0.0011 | Validation Loss: 0.0009
Epoch [5/10] - Train Loss: 0.0015 | Validation Loss: 0.0005
Epoch [6/10] - Train Loss: 0.0014 | Validation Loss: 0.0004
Epoch [7/10] - Train Loss: 0.0008 | Validation Loss: 0.0004
Epoch [8/10] - Train Loss: 0.0008 | Validation Loss: 0.0006
Epoch [9/10] - Train Loss: 0.0007 | Validation Loss: 0.0007
Epoch [10/10] - Train Loss: 0.0011 | Validation Loss: 0.0003
Training complete! Model saved as '30.4.26.pth'


Driving

In [ ]:
import ipywidgets.widgets as widgets
from IPython.display import display
import cv2
import numpy as np
import pyzed.sl as sl
import threading
import traitlets
from traitlets.config.configurable import SingletonConfigurable
import time
import torch
import torchvision
import torchvision.transforms as transforms
from PIL import Image

# Initialise Motors
import motors
robot = motors.MotorsYukon(mecanum=False)

# Setup Neural Network for Inference
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = torchvision.models.resnet18(weights=None) 
model.fc = torch.nn.Linear(512, 2) # 2 outputs: x and y

# Load the trained weights
model.load_state_dict(torch.load('30.4.26_third.pth', map_location=device)) 
model = model.to(device)
model.eval() 

# Transform to match training
transform = transforms.Compose([
    transforms.Resize((224, 224), antialias=True),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Setup UI
display_img = widgets.Image(format='jpeg', width='50%')
display(display_img)

def bgr8_to_jpeg(value):
    return bytes(cv2.imencode('.jpg', value)[1])

# ZED Camera Initialisation with AVI Recording
class Camera(SingletonConfigurable):
    color_value = traitlets.Any() 
    
    def __init__(self):
        super(Camera, self).__init__()
        self.zed = sl.Camera()
        init_params = sl.InitParameters()
        init_params.camera_resolution = sl.RESOLUTION.VGA 
        init_params.depth_mode = sl.DEPTH_MODE.ULTRA 
        init_params.coordinate_units = sl.UNIT.MILLIMETER  

        status = self.zed.open(init_params)
        if status != sl.ERROR_CODE.SUCCESS:
            print("Camera Open : "+repr(status)+". Exit program.")
            self.zed.close()
            exit(1)

        self.runtime = sl.RuntimeParameters()
        self.thread_running_flag = False
        camera_info = self.zed.get_camera_information()
        self.width = camera_info.camera_configuration.resolution.width
        self.height = camera_info.camera_configuration.resolution.height
        
        fps = camera_info.camera_configuration.fps
        fourcc = cv2.VideoWriter_fourcc(*'XVID') 
        self.video_writer = cv2.VideoWriter('color_video_f320951_9.avi', fourcc, fps, (self.width, self.height))
        print(f"Recording Started: color_video_f320951_9.avi at {fps} FPS")

        self.image = sl.Mat(self.width, self.height, sl.MAT_TYPE.U8_C4, sl.MEM.CPU)

    def _capture_frames(self):
        start_time = time.time()
        
        while(self.thread_running_flag == True):
            # Record for 120 seconds, then stop and save the AVI file
            if time.time() - start_time >= 120.0:
                print("\n120 seconds reached. Stopping robot and saving AVI video...")
                self.thread_running_flag = False
                break
                
            if self.zed.grab(self.runtime) == sl.ERROR_CODE.SUCCESS:
                self.zed.retrieve_image(self.image, sl.VIEW.LEFT)
                color_value_BGRA = self.image.get_data()
                # Convert to standard BGR for OpenCV
                self.color_value = cv2.cvtColor(color_value_BGRA, cv2.COLOR_BGRA2BGR)
                # Write this specific frame to the .avi file
                self.video_writer.write(self.color_value)
            time.sleep(0.001)
            
        self.video_writer.release()
        self.zed.close()
        robot.stop() 
        print("AVI file saved.")
                
    def start(self): 
        if self.thread_running_flag == False: 
            self.thread_running_flag = True 
            self.thread = threading.Thread(target=self._capture_frames) 
            self.thread.start() 

    def stop(self): 
        if self.thread_running_flag == True:
            self.thread_running_flag = False 
            self.thread.join()

# Smooth Turning Function
def smooth_turn(robot_instance, direction):
    """
    Executes a smooth forward arc by driving all wheels forward, 
    by spinning the outside wheels faster than the inside wheels.
    """
    # Hardcoded turning speeds (Larger difference = sharper turn, smaller difference = wider turn)
    fast_speed = 0.6  # Outside wheels 
    slow_speed = 0.0  # Inside wheels 
    
    if direction == "left":
        # Right wheels go fast, left wheels go slow
        message = (
            f"m;direction:frontleft;speed:{slow_speed}\n"
            f"m;direction:backleft;speed:{slow_speed}\n"
            f"m;direction:frontright;speed:{fast_speed}\n"
            f"m;direction:backright;speed:{fast_speed}\n"
        )
        robot_instance.send(message)
        
    elif direction == "right":
        # Left wheels go fast, right wheels go slow
        message = (
            f"m;direction:frontleft;speed:{fast_speed}\n"
            f"m;direction:backleft;speed:{fast_speed}\n"
            f"m;direction:frontright;speed:{slow_speed}\n"
            f"m;direction:backright;speed:{slow_speed}\n"
        )
        robot_instance.send(message)

# Inference and Control Loop
def process_vision(change):
    frame = change['new'].copy()
    # Prepare image for the network
    img_pil = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    tensor_img = transform(img_pil).unsqueeze(0).to(device)
    
    # Predict the path coordinates
    with torch.no_grad():
        output = model(tensor_img)
        x_pred = output[0][0].item() # Predicted X (-1.0 to 1.0)
        y_pred = output[0][1].item() # Predicted Y (-1.0 to 1.0)
    
    height, width, _ = frame.shape
    turn_threshold = 0.15 # Center deadzone
    
    # Calculate pixel coordinates for the Left and Right thresholds
    left_line_px = int((-turn_threshold + 1.0) / 2.0 * width)
    right_line_px = int((turn_threshold + 1.0) / 2.0 * width)
    # Draw the vertical threshold lines (Yellow)
    cv2.line(frame, (left_line_px, 0), (left_line_px, height), (0, 255, 255), 2)
    cv2.line(frame, (right_line_px, 0), (right_line_px, height), (0, 255, 255), 2)

    # Draw a static blue dot in the exact center of the frame 
    center_x = width // 2
    center_y = height // 2
    cv2.circle(frame, (center_x, center_y), 8, (255, 0, 0), -1) 
    # Draw a green dot showing where the network predicts the line is
    px = int((x_pred + 1.0) / 2.0 * width)
    py = int((y_pred + 1.0) / 2.0 * height)
    cv2.circle(frame, (px, py), 12, (0, 255, 0), -1)
    
    # Control strategy based on predicted x coordinate
    if x_pred < -turn_threshold:
        smooth_turn(robot, "left")
        cv2.putText(frame, "SMOOTH LEFT", (50, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)
        
    elif x_pred > turn_threshold:
        smooth_turn(robot, "right")
        cv2.putText(frame, "SMOOTH RIGHT", (50, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)
        
    else:
        robot.forward(0.3) 
        cv2.putText(frame, "FORWARD", (50, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

    # Update Widget
    scale = 0.5
    resized = cv2.resize(frame, None, fx=scale, fy=scale)
    display_img.value = bgr8_to_jpeg(resized)

# Start the System
camera = Camera()
camera.observe(process_vision, names=['color_value'])
camera.start()

/tmp/ipykernel_2899/1873874244.py:25: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load('30.4.26_third.pth', map_location=device))


Image(value=b'', format='jpeg', width='50%')

[2026-05-08 08:40:02 UTC][ZED][INFO] Logging level INFO
[2026-05-08 08:40:02 UTC][ZED][INFO] Logging level INFO
Recording Started: color_video_f320951_9.svo2
[2026-05-08 08:40:02 UTC][ZED][INFO] Logging level INFO
[2026-05-08 08:40:03 UTC][ZED][INFO] [Init]  Depth mode: ULTRA
[2026-05-08 08:40:04 UTC][ZED][INFO] [Init]  Camera successfully opened.
[2026-05-08 08:40:04 UTC][ZED][INFO] [Init]  Camera FW version: 1523
[2026-05-08 08:40:04 UTC][ZED][INFO] [Init]  Video mode: VGA@100
[2026-05-08 08:40:04 UTC][ZED][INFO] [Init]  Serial Number: S/N 35860577
[2026-05-08 08:40:04 UTC][ZED][INFO] [Init]  Notice: The recording is using SVO version 2, enabled by default starting from SDK version 4.1. To revert to the original SVO version, set the environment variable "ZED_SDK_SVO_VERSION" to 1


In [2]:
import motors

robot = motors.MotorsYukon(mecanum=False)
robot.stop()